**Run note:** execute this notebook's first setup/code cell before any later cells. Each notebook is designed to run independently and re-detect the dataset path on its own.

# 15 — Model Packaging & REST API

Package everything needed for inference and expose a FastAPI endpoint:
1. Save model weights + processor + metadata as a deployment bundle
2. Write a single-file inference helper
3. Serve a `/predict` endpoint via FastAPI + Uvicorn + pyngrok (public URL on Kaggle)

In [ ]:
# Install serving stack (adds ~1 min on first run)
import subprocess
for pkg in ["fastapi", "uvicorn[standard]", "python-multipart", "pyngrok"]:
    subprocess.run(["pip", "install", "-q", pkg], check=True)
print("Packages ready.")

In [ ]:
import os
import json
import re
import shutil
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

ON_KAGGLE = Path("/kaggle/input").is_dir()
JSONL_CANDIDATES = {
    "train": ["train.jsonl"],
    "dev": ["dev.jsonl", "dev_seen.jsonl", "dev_unseen.jsonl"],
    "test": ["test.jsonl", "test_seen.jsonl", "test_unseen.jsonl"],
}
IMAGE_DIR_CANDIDATES = ("img", "images")


def _has_image_dir(path: Path) -> bool:
    return any((path / name).is_dir() for name in IMAGE_DIR_CANDIDATES)


def _has_any_jsonl(path: Path, names) -> bool:
    return any((path / name).is_file() for name in names)


def _looks_like_dataset_root(path: Path) -> bool:
    return path.is_dir() and _has_image_dir(path) and _has_any_jsonl(path, JSONL_CANDIDATES["train"])


def detect_data_dir():
    for env_name in ("KAGGLE_DATA_DIR", "META_HATEFUL_MEME_DATA_DIR"):
        env_dir = os.environ.get(env_name, "").strip()
        if env_dir and _looks_like_dataset_root(Path(env_dir)):
            return Path(env_dir), f"env:{env_name}"

    kaggle_input = Path("/kaggle/input")
    default_candidate = kaggle_input / "meta-hateful-meme-detection" / "data"
    if _looks_like_dataset_root(default_candidate):
        return default_candidate, "default:/kaggle/input/meta-hateful-meme-detection/data"

    if ON_KAGGLE:
        for train_jsonl in sorted(kaggle_input.rglob("train.jsonl")):
            candidate = train_jsonl.parent
            if _looks_like_dataset_root(candidate):
                return candidate, f"auto:{candidate}"

        for candidate in sorted(kaggle_input.rglob("*")):
            if candidate.is_dir() and _looks_like_dataset_root(candidate):
                return candidate, f"auto:{candidate}"

    for candidate in (Path.cwd() / "data", Path.cwd().parent / "data", Path.cwd(), Path.cwd().parent):
        if _looks_like_dataset_root(candidate):
            return candidate, f"local:{candidate}"

    return None, "not-found"


DATA_DIR, data_source = detect_data_dir()
if DATA_DIR is None:
    raise FileNotFoundError(
        "Dataset not found. Set KAGGLE_DATA_DIR or META_HATEFUL_MEME_DATA_DIR to the folder containing train.jsonl and img/."
    )

IMG_DIR = next((DATA_DIR / name for name in IMAGE_DIR_CANDIDATES if (DATA_DIR / name).is_dir()), None)
OUTPUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path.cwd() / "outputs"
BUNDLE_DIR = OUTPUT_DIR / "model_bundle"
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_DIR = str(DATA_DIR)
IMG_DIR = str(IMG_DIR) if IMG_DIR is not None else None
OUTPUT_DIR = str(OUTPUT_DIR)
BUNDLE_DIR = str(BUNDLE_DIR)

print(f"Using DATA_DIR : {DATA_DIR}")
print(f"Using IMG_DIR  : {IMG_DIR}")
print(f"Using source   : {data_source}")
print(f"Output dir     : {OUTPUT_DIR}")
print(f"Bundle dir     : {BUNDLE_DIR}")
print(f"Device         : {DEVICE}")

# Model classes (inline)
class CLIPEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
        for p in self.clip.parameters():
            p.requires_grad_(False)

    def forward(self, pv, ids, mask):
        return (
            F.normalize(self.clip.get_image_features(pixel_values=pv), -1),
            F.normalize(self.clip.get_text_features(input_ids=ids, attention_mask=mask), -1),
        )


class CrossAttentionFusion(nn.Module):
    def __init__(self, d=512, h=4):
        super().__init__()
        self.i2t = nn.MultiheadAttention(d, h, batch_first=True)
        self.t2i = nn.MultiheadAttention(d, h, batch_first=True)
        self.ni = nn.LayerNorm(d)
        self.nt = nn.LayerNorm(d)

    def forward(self, i, t):
        is_ = i.unsqueeze(1)
        ts = t.unsqueeze(1)
        ic, ia = self.i2t(is_, ts, ts)
        tc, ta = self.t2i(ts, is_, is_)
        return torch.cat([self.ni(i + ic.squeeze(1)), self.nt(t + tc.squeeze(1))], -1), ia, ta


class HatefulMemeClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = CLIPEncoder()
        self.fusion = CrossAttentionFusion()
        self.head = nn.Sequential(
            nn.Linear(1024, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2),
        )

    def forward(self, pv, ids, mask):
        i, t = self.encoder(pv, ids, mask)
        f, ia, ta = self.fusion(i, t)
        return self.head(f), ia, ta

In [ ]:
# â”€â”€ 14.1 Package model bundle â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model     = HatefulMemeClassifier().to(DEVICE)
ckpt      = os.path.join(OUTPUT_DIR, "cross_attention_best.pt")

if os.path.exists(ckpt):
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    print("Loaded trained checkpoint.")
else:
    print("WARNING: using random weights â€” run notebook 08 first.")

model.eval()

# Save weights + metadata
torch.save(model.state_dict(), os.path.join(BUNDLE_DIR, "weights.pt"))
processor.save_pretrained(BUNDLE_DIR)

meta = {
    "model": "HatefulMemeClassifier",
    "clip_backbone": "openai/clip-vit-base-patch32",
    "fusion": "CrossAttentionFusion",
    "embed_dim": 512,
    "num_classes": 2,
    "labels": {"0": "non_hateful", "1": "hateful"},
    "threshold": 0.5,
    "max_text_len": 77,
    "image_size": 224,
}
with open(os.path.join(BUNDLE_DIR, "meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print(f"Bundle saved to {BUNDLE_DIR}")
print(os.listdir(BUNDLE_DIR))

In [ ]:
# â”€â”€ 14.2 Inference helper â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def clean_text(text):
    if not isinstance(text, str): return "[no text]"
    return re.sub(r"\s+"," ", re.sub(r"@\w+"," ", re.sub(r"https?://\S+"," ",text))).strip() or "[no text]"

@torch.no_grad()
def predict_from_pil(pil_image: Image.Image, text: str, threshold: float = 0.5):
    """End-to-end prediction from a PIL image + text string."""
    text = clean_text(text)
    enc  = processor(
        text=[text], images=pil_image.convert("RGB"),
        return_tensors="pt", padding="max_length",
        max_length=77, truncation=True
    )
    pv   = enc["pixel_values"].to(DEVICE)
    ids  = enc["input_ids"].to(DEVICE)
    mask = enc["attention_mask"].to(DEVICE)

    logits, img_attn, txt_attn = model(pv, ids, mask)
    probs = torch.softmax(logits, -1)[0].cpu().numpy()
    label = int(probs[1] >= threshold)

    return {
        "label": "hateful" if label == 1 else "non_hateful",
        "label_id": label,
        "confidence": float(probs[label]),
        "p_hateful": float(probs[1]),
        "p_non_hateful": float(probs[0]),
    }

# Quick smoke test on a dev image
test_row = json.loads(open(os.path.join(DATA_DIR, "dev.jsonl")).readline())
img_path = resolve_image_path(DATA_DIR, test_row["img"])
if os.path.exists(img_path):
    result = predict_from_pil(Image.open(img_path), test_row["text"])
    print("Smoke test:", result)
    print(f"Ground truth: {test_row.get('label', 'unknown')}")

In [ ]:
# â”€â”€ 14.3 Write inference script â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
inference_script = '''
"""
infer.py â€” standalone inference script for HatefulMemeClassifier
Usage:  python infer.py --img path/to/image.jpg  [--text "meme text"]
        python infer.py --bundle /path/to/model_bundle
"""
import argparse, json, os, re
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

class CLIPEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.clip=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
        for p in self.clip.parameters(): p.requires_grad_(False)
    def forward(self,pv,ids,mask):
        return (F.normalize(self.clip.get_image_features(pixel_values=pv),-1),
                F.normalize(self.clip.get_text_features(input_ids=ids,attention_mask=mask),-1))

class CrossAttentionFusion(nn.Module):
    def __init__(self,d=512,h=4):
        super().__init__()
        self.i2t=nn.MultiheadAttention(d,h,batch_first=True); self.t2i=nn.MultiheadAttention(d,h,batch_first=True)
        self.ni=nn.LayerNorm(d); self.nt=nn.LayerNorm(d)
    def forward(self,i,t):
        is_=i.unsqueeze(1); ts=t.unsqueeze(1)
        ic,ia=self.i2t(is_,ts,ts); tc,ta=self.t2i(ts,is_,is_)
        return torch.cat([self.ni(i+ic.squeeze(1)),self.nt(t+tc.squeeze(1))],-1),ia,ta

class HatefulMemeClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=CLIPEncoder(); self.fusion=CrossAttentionFusion()
        self.head=nn.Sequential(nn.Linear(1024,256),nn.GELU(),nn.Dropout(0.3),
                                  nn.Linear(256,128),nn.GELU(),nn.Dropout(0.3),nn.Linear(128,2))
    def forward(self,pv,ids,mask):
        i,t=self.encoder(pv,ids,mask); f,ia,ta=self.fusion(i,t); return self.head(f),ia,ta


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--img", required=True)
    parser.add_argument("--text", default="")
    parser.add_argument("--bundle", default="./model_bundle")
    parser.add_argument("--threshold", type=float, default=0.5)
    args = parser.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    proc   = CLIPProcessor.from_pretrained(args.bundle)
    model  = HatefulMemeClassifier().to(device)
    model.load_state_dict(torch.load(os.path.join(args.bundle,"weights.pt"), map_location=device))
    model.eval()

    text = re.sub(r"\\s+"," ", args.text).strip() or "[no text]"
    img  = Image.open(args.img).convert("RGB")
    enc  = proc(text=[text], images=img, return_tensors="pt",
                padding="max_length", max_length=77, truncation=True)

    with torch.no_grad():
        logits,_,_ = model(enc["pixel_values"].to(device),
                           enc["input_ids"].to(device),
                           enc["attention_mask"].to(device))
        probs = torch.softmax(logits,-1)[0].cpu().numpy()

    label = "hateful" if probs[1] >= args.threshold else "non_hateful"
    print(json.dumps({"label": label, "p_hateful": float(probs[1]),
                      "p_non_hateful": float(probs[0])}, indent=2))

if __name__ == "__main__":
    main()
'''

with open(os.path.join(OUTPUT_DIR, "infer.py"), "w") as f:
    f.write(inference_script.strip())
print("infer.py written.")

In [ ]:
# â”€â”€ 14.4 FastAPI app â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
api_code = '''
"""FastAPI app for hateful meme detection."""
from fastapi import FastAPI, File, UploadFile, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import torch, torch.nn as nn, torch.nn.functional as F
import re, io
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

# â”€â”€ Model definitions (inline) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
class CLIPEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.clip=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
        for p in self.clip.parameters(): p.requires_grad_(False)
    def forward(self,pv,ids,mask):
        return (F.normalize(self.clip.get_image_features(pixel_values=pv),-1),
                F.normalize(self.clip.get_text_features(input_ids=ids,attention_mask=mask),-1))

class CrossAttentionFusion(nn.Module):
    def __init__(self,d=512,h=4):
        super().__init__()
        self.i2t=nn.MultiheadAttention(d,h,batch_first=True); self.t2i=nn.MultiheadAttention(d,h,batch_first=True)
        self.ni=nn.LayerNorm(d); self.nt=nn.LayerNorm(d)
    def forward(self,i,t):
        is_=i.unsqueeze(1); ts=t.unsqueeze(1)
        ic,ia=self.i2t(is_,ts,ts); tc,ta=self.t2i(ts,is_,is_)
        return torch.cat([self.ni(i+ic.squeeze(1)),self.nt(t+tc.squeeze(1))],-1),ia,ta

class HatefulMemeClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=CLIPEncoder(); self.fusion=CrossAttentionFusion()
        self.head=nn.Sequential(nn.Linear(1024,256),nn.GELU(),nn.Dropout(0.3),
                                  nn.Linear(256,128),nn.GELU(),nn.Dropout(0.3),nn.Linear(128,2))
    def forward(self,pv,ids,mask):
        i,t=self.encoder(pv,ids,mask); f,ia,ta=self.fusion(i,t); return self.head(f),ia,ta

DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
BUNDLE    = "/kaggle/working/model_bundle"
processor = CLIPProcessor.from_pretrained(BUNDLE)
model     = HatefulMemeClassifier().to(DEVICE)
model.load_state_dict(torch.load(f"{BUNDLE}/weights.pt", map_location=DEVICE))
model.eval()

app = FastAPI(title="Hateful Meme Detector", version="1.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class PredictionResponse(BaseModel):
    label: str
    label_id: int
    confidence: float
    p_hateful: float
    p_non_hateful: float

@app.get("/health")
def health(): return {"status": "ok", "device": DEVICE}

@app.post("/predict", response_model=PredictionResponse)
async def predict(
    image: UploadFile = File(...),
    text:  str        = Form(default=""),
    threshold: float  = Form(default=0.5),
):
    if not image.content_type.startswith("image/"):
        raise HTTPException(400, "File must be an image")
    if not 0.0 < threshold < 1.0:
        raise HTTPException(400, "threshold must be between 0 and 1")

    raw = await image.read()
    img = Image.open(io.BytesIO(raw)).convert("RGB")
    txt = re.sub(r"\\s+"," ", text).strip() or "[no text]"

    enc = processor(text=[txt], images=img, return_tensors="pt",
                    padding="max_length", max_length=77, truncation=True)
    with torch.no_grad():
        logits,_,_ = model(enc["pixel_values"].to(DEVICE),
                           enc["input_ids"].to(DEVICE),
                           enc["attention_mask"].to(DEVICE))
        probs = torch.softmax(logits,-1)[0].cpu().numpy()

    lbl = int(probs[1] >= threshold)
    return PredictionResponse(
        label="hateful" if lbl else "non_hateful",
        label_id=lbl,
        confidence=float(probs[lbl]),
        p_hateful=float(probs[1]),
        p_non_hateful=float(probs[0]),
    )
'''

with open(os.path.join(OUTPUT_DIR, "api.py"), "w") as f:
    f.write(api_code.strip())
print("api.py written.")

In [ ]:
# â”€â”€ 14.5 Launch FastAPI with pyngrok public URL â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import threading, time, subprocess
from pyngrok import ngrok

PORT = 8000

def run_server():
    subprocess.run(["uvicorn", "api:app", "--host", "0.0.0.0",
                    "--port", str(PORT), "--no-access-log"],
                   cwd=OUTPUT_DIR)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(3)  # wait for server to start

public_url = ngrok.connect(PORT)
print("\n" + "=" * 55)
print(f"FastAPI running at: {public_url}")
print(f"  Health: {public_url}/health")
print(f"  Docs  : {public_url}/docs")
print("  POST /predict  (form-data: image + text + threshold)")
print("=" * 55)

In [ ]:
# â”€â”€ 14.6 Test the API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import requests

# Use first dev image as test
import json
test_row = json.loads(open(os.path.join(DATA_DIR, "dev.jsonl")).readline())
img_path = os.path.join(DATA_DIR, test_row["img"])

if os.path.exists(img_path):
    with open(img_path, "rb") as img_file:
        response = requests.post(
            f"http://localhost:{PORT}/predict",
            files={"image": ("test.jpg", img_file, "image/jpeg")},
            data={"text": test_row.get("text", ""), "threshold": "0.5"}
        )
    print("API response:", json.dumps(response.json(), indent=2))
    print(f"Ground truth: {test_row.get('label', 'unknown')}")
else:
    print("Test image not found â€” skip.")

In [ ]:
# â”€â”€ Summary â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print("=" * 55)
print("PACKAGING & API SUMMARY")
print("=" * 55)
print(f"Model bundle saved to : {BUNDLE_DIR}")
print(f"Inference script      : {OUTPUT_DIR}/infer.py")
print(f"FastAPI app           : {OUTPUT_DIR}/api.py")
print()
print("Endpoints:")
print("  GET  /health   â€” liveness check")
print("  GET  /docs     â€” interactive Swagger UI")
print("  POST /predict  â€” image (file) + text (form) â†’ label + confidence")
print()
print("Security notes:")
print("  - Image content-type validated before loading")
print("  - Threshold range validated [0, 1]")
print("  - CORS set to allow all origins (restrict in production)")
print("\nProceed to notebook 15 (Gradio Demo).")